<a href="https://colab.research.google.com/github/jacbarbosa279/mbaproject/blob/dev/vocal_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa soundfile numpy scipy noisereduce

In [3]:
import librosa
import numpy as np
import soundfile as sf
import noisereduce as nr
from scipy.ndimage import median_filter, gaussian_filter

# Extensões vocais aproximadas em Hertz (Min e Max)
VOCAL_RANGES = {
    "baixo": (82.0, 330.0),
    "tenor": (130.0, 523.0),
    "contralto": (174.0, 698.0),
    "soprano": (261.0, 1046.0)
}

def isolate_vocal_part(y, sr, fmin, fmax, num_harmonics=8):
    """
    Isola dinamicamente um naipe vocal preservando seus harmônicos para evitar som abafado.
    """
    # 1. Transformada de Fourier (STFT) de alta resolução
    n_fft = 4096
    hop_length = 1024
    S_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    S_mag, S_phase = librosa.magphase(S_complex)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    # Matriz vazia para a máscara
    mask = np.zeros_like(S_mag)

    # Índices das frequências fundamentais do naipe
    fund_bins = np.where((freqs >= fmin) & (freqs <= fmax))[0]

    # Nível médio de energia para ignorar silêncio
    energy_threshold = np.mean(S_mag) * 0.5

    # 2. Rastreamento e Máscara Harmônica (Frame a Frame)
    for t in range(S_mag.shape[1]):
        frame_mag = S_mag[:, t]
        fund_region = frame_mag[fund_bins]

        # Se houver som cantado neste instante de tempo
        if np.max(fund_region) > energy_threshold:
            # Encontra o pico de frequência (a nota dominante) do naipe
            local_peak_idx = np.argmax(fund_region)
            global_peak_idx = fund_bins[local_peak_idx]
            f0 = freqs[global_peak_idx]

            # Libera a frequência fundamental e seus harmônicos
            for h in range(1, num_harmonics + 1):
                fh = f0 * h
                if fh < sr / 2: # Respeita o limite de Nyquist
                    h_idx = np.argmin(np.abs(freqs - fh))

                    # Calcula uma pequena margem (bandwith) ao redor do harmônico para o vibrato
                    bw = max(1, int(h_idx * 0.02)) # 2% de largura

                    # Preenche a máscara com 1.0 (som passa 100%) nessa janela
                    mask[max(0, h_idx - bw) : min(len(freqs), h_idx + bw + 1), t] = 1.0

    # 3. Suavização da Máscara (Evita sons robóticos e cliques)
    # Filtro mediano tira ruídos pontuais, filtro gaussiano suaviza a entrada/saída do som
    mask = median_filter(mask, size=(3, 3))
    mask = gaussian_filter(mask, sigma=1.0)

    # 4. Aplicação e Reconstrução
    S_separated = S_mag * mask
    y_separated = librosa.istft(S_separated * np.exp(1j * S_phase), hop_length=hop_length)

    return y_separated

def process_choir_audio(input_file):
    print(f"Carregando {input_file} (Alta qualidade)...")
    y, sr = librosa.load(input_file, sr=44100, mono=True)

    for voice, (fmin, fmax) in VOCAL_RANGES.items():
        print(f"Processando {voice.upper()}...")

        # Isola a voz preservando os harmônicos (evita som abafado)
        y_isolated = isolate_vocal_part(y, sr, fmin, fmax)

        # Aplica Spectral Gating profundo para remover chiados e vazamentos (sem ruído)
        print(f"  -> Removendo ruídos de {voice}...")
        y_clean = nr.reduce_noise(y=y_isolated, sr=sr, prop_decrease=0.85, stationary=False)

        # Normaliza o volume para que fique audível e alto
        if np.max(np.abs(y_clean)) > 0:
            y_clean = y_clean / np.max(np.abs(y_clean))

        output_name = f"{voice}_cristalino.wav"
        sf.write(output_name, y_clean, sr)
        print(f"  -> Salvo: {output_name}")

# === COMO EXECUTAR ===
if __name__ == "__main__":
    # Definido para o arquivo correto fornecido no ambiente
    arquivo = "/content/vocals.wav"

    import os
    if os.path.exists(arquivo):
        process_choir_audio(arquivo)
        print("\nSeparação concluída com sucesso! Os áudios não terão chiado e manterão o brilho original.")
    else:
        print(f"Erro: Faça o upload do arquivo '{arquivo}' no Colab antes de rodar.")

Carregando /content/vocals.wav (Alta qualidade)...
Processando BAIXO...
  -> Removendo ruídos de baixo...


/usr/local/lib/python3.13/dist-packages/noisereduce/spectralgate/nonstationary.py:70: RuntimeWarning: invalid value encountered in divide
  sig_mult_above_thresh = (abs_sig_stft - sig_stft_smooth) / sig_stft_smooth


  -> Salvo: baixo_cristalino.wav
Processando TENOR...
  -> Removendo ruídos de tenor...
  -> Salvo: tenor_cristalino.wav
Processando CONTRALTO...
  -> Removendo ruídos de contralto...
  -> Salvo: contralto_cristalino.wav
Processando SOPRANO...
  -> Removendo ruídos de soprano...
  -> Salvo: soprano_cristalino.wav

Separação concluída com sucesso! Os áudios não terão chiado e manterão o brilho original.
